In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.2 MB/s eta 0:00:00


In [2]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [3]:
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [4]:
audio_filename = "/content/king_county_meeting.mp3"

In [7]:
hf_token = userdata.get('HF_Token')
login(hf_token, add_to_git_credential=True)

In [8]:
audio_file = open(audio_filename, "rb")

### Open source transcription use

In [9]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


 Good morning and welcome to the February 7th 2018 meeting of the King County Council Committee of the Whole Today the committee will be discussing the second of four ordinances to make the King County Code gender neutral And then we will be going into an extended studies executive session for two items or one regarding the Convention Center sale The other regarding the implementation of ordinance one eight four zero three which set compensation fees and costs for using county right-of-way and we will probably not come I'm predicting at this point we will not come out of executive session I will just adjourn from executive session so those are the topics on the agenda for today I will skip the roll until we have a few more people and go right to public comment I think as you all know the Committee of the Whole offers the public the opportunity to make comments on any item on today's agenda I've just talked through what those are each person will have two minutes to speak public comment

In [10]:
"abc"

'abc'

In [11]:
open_source_transcription = transcription

In [12]:
open_source_transcription

" Good morning and welcome to the February 7th 2018 meeting of the King County Council Committee of the Whole Today the committee will be discussing the second of four ordinances to make the King County Code gender neutral And then we will be going into an extended studies executive session for two items or one regarding the Convention Center sale The other regarding the implementation of ordinance one eight four zero three which set compensation fees and costs for using county right-of-way and we will probably not come I'm predicting at this point we will not come out of executive session I will just adjourn from executive session so those are the topics on the agenda for today I will skip the roll until we have a few more people and go right to public comment I think as you all know the Committee of the Whole offers the public the opportunity to make comments on any item on today's agenda I've just talked through what those are each person will have two minutes to speak public commen

### Frontier model : transcription use through OpenAI

In [13]:
AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

Good morning and welcome to the February 7th, 2018 meeting of the King County Council Committee of the Whole. Today, the committee will be discussing the second of four ordinances to make the King County Code gender neutral, and then we will be going into an extended studies executive session for two items, one regarding the convention center sale, the other regarding the implementation of Ordinance 18403, which set compensation fees and costs for using county right-of-way. And we will probably not come, I'm predicting at this point, we will not come out of executive session, we'll just adjourn from executive session. So those are the topics on the agenda for today. I will skip the role until we have a few more people, and go right to public comment. I think as you all know, the Committee of the Whole offers the public the opportunity to make comments on any item on today's agenda. I've just talked through what those are. Each person will have two minutes to speak. Public comment must 

In [14]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

 Good morning and welcome to the February 7th 2018 meeting of the King County Council Committee of the Whole Today the committee will be discussing the second of four ordinances to make the King County Code gender neutral And then we will be going into an extended studies executive session for two items or one regarding the Convention Center sale The other regarding the implementation of ordinance one eight four zero three which set compensation fees and costs for using county right-of-way and we will probably not come I'm predicting at this point we will not come out of executive session I will just adjourn from executive session so those are the topics on the agenda for today I will skip the roll until we have a few more people and go right to public comment I think as you all know the Committee of the Whole offers the public the opportunity to make comments on any item on today's agenda I've just talked through what those are each person will have two minutes to speak public comment must address an item on today's agenda public comment may not be used for the purpose of assisting a campaign for election of any person or for the promotion or opposition of any ballot proposition it must not include obscene speech if a speaker fails to abide by these restrictions I will rule them out of order require them to return to their seat so with that we have one person signed up to give public comment and And that's Mr. Alex Zimmerman. Good morning, and welcome. Hi. My name is Dori W. Fuhrer, a Nazi social democratic mafia with progressive Gestapo principle. My name is Alex Zimmerman, and I am president of Stand Up America. I want to speak about this agenda, because agenda just make me absolutely understandable. So you right now change from vagina and penis to somebody who like an ordinary man, I'm totally confused about this. It's a huge ordinary, like a hundred probably, yeah, it's good changes. My question right now, very simple, when you have fundamental changes like this, why have fundamental changes for 30 percentage taxes, whereas we have only for one calendar year now? I don't think so, this happened before. So when we change from men to women to something, people, you know what I mean, it's okay with me, but how we can change the 30 percentage taxes is I cannot understand this. If you're talking about billion and billion dollars, so when you pay right now for sound transits, hundred billion dollars pounds is here. When you pay right now for teacher five billion dollars, another not real changes. In Boeing, nine billion dollars right now, in another billion and billion dollars, you never come change this to normal situation. When you change from man to woman into people, I understand this, but when you cannot change hundred billion dollars what is you stolen from us, like a criminal, like a gangster, like a killer, I cannot understand this. Small change is very good too. But we need right now come back to America, what is my President Donald Trump talking about this. And he call you a low life. Mr. Zimmerman, are you going to speak to something on the agenda today? Exactly I speak about this. I'm not hearing a single thing that's related to anything that's on the agenda today. No, exactly I speak, you have too many changes, but you never have a fundamental change what is make us life better. Fundamental change that make our life better is not on the agenda today. Stand up America. You're out of time, thank you. All right, I understand that we have one other person who would like to speak and that's Kasit Zenebi, welcome. I'll ask you all, so please speak to something on the agenda today. That's what public comment is for, so if you could please try to speak to something on our agenda. And the items are about gender neutral code, the convention center sale, and compensation for fees and costs using King County right of way. Thank you. On a single day, the United States, my TC, advised that in one place, we'll in another location, people are enjoying a warm sunny afternoon in general. The United States is excited to have a temperature climate, meaning the weather, the virus, over the four seasons. are warm and the winter is cold, but they're in a great rearrange with the poor climate, but part of current, with cold Mediterranean climate, with hot, dry summers, and with cold winter part flood area and consider tropic with warm temperature and the higher and detail all here in general temperature are colder in the north part of country in where most the south but even thank you thank you all right we're gonna move on we'll also pass over item four and approve the minutes when we have a quorum and move to item five which is proposed ordinance number 2018 0 0 6 8 it's our continuing work to establish a gender neutral language in our code. This one has a number of titles which I assume will be listed out. I have them here but I'm not gonna list them out. And it's the second of four planned ordinances that together will make the entire King County Code gender neutral as we last year had the went to the voters to make the King County Charter gender neutral. We have King County Central staff Erin Ozzens here to present. Welcome and please go ahead and give your presentation good morning Aaron odds and council staff the materials for this item begin on page 9 of your packet proposed ordinance 2018 does 0 0 6 8 is the second in a series of four ordinances that would make changes to the King County Code to remove gendered pronouns and historically gendered terms wherever possible this ordinances this ordinance includes the titles of the King County Code that collectively are the county's Development Regulations, which do include Titles 9, 13, 14, 16, 17, 19A, 20, 21A, 23, 27, and 27A. No substantive legal or policy changes are proposed to be made as part of this ordinance, but other drafting corrections are proposed by the Code Advisor and have been incorporated into the proposed ordinance. By way of background, in July 2016, Motion 14680 was passed by the Council, directing the clerk of the council to develop options for how to apply gender neutral preferences throughout the King County Code. The same day, related ordinance 18316 placed an item on the ballot to amend the charter to make the language of the charter gender neutral. That charter amendment passed by a majority of the voters in November of 2016. This proposed ordinance is consistent with existing Washington state law and county code to be written in gender-neutral terms. In the proposed ordinance, gendered pronouns, such as he, him, she, or her, are replaced with the title of the actor in impacted sentences. For example, in sections that refer to the director as he or she, the proposed ordinance changes the gendered pronoun to the director, thereby naming the title of the actor and disregarding the gender. Table 1 on page 10 of the packet contains a sample of other proposed changes to historically gendered terms. A comprehensive list of the gender terms addressed in the proposed ordinance is available in attachment two of the staff report which starts on page 159 of the packet. Executive staff were consulted regarding the proposed changes and their feedback has been addressed and incorporated into the proposed ordinance. In attachment three on page 163 of your packet is the timeline for the future ordinances. There are two additional ordinances that will be presented to the council and that will then complete the review of the remaining titles of the code. The entire project should be complete by May of 2018. That concludes my report. As with the previous ordinance, it's important to recognize the work of Russell Pethel and Bruce Ritzen from the clerk's office on this project. Thank you. Thank you, Erin, and you as well. The staff has been doing an amazing job on this. It's a lot of detail work, but I think it's really, we're gonna have a better code at the end of it. I really appreciated the portion of your report that talked about some of the detail that goes into this the practicality where we had proposed changing manhole as in manhole covers to like utility access or something point but apparently out on the streets and in documents it's there are abbreviations all over the place that say MH so we came up with maintenance hole so that now people will know what MH means even example of incorporating the executive staff's comments that was their request so I appreciate the level of effort that's going into making this work well Any questions or comments? Council Member Colwells. Thank you, Madam Chair and all the staff. Thank you for all your excellent work on this. Two things I'd like to bring up. On the top of page 10, just so we'll have accurate information here, on the second line, additionally in in 2007, basically the state legislature passed Senate Bill 5063. That was the first year of I believe it was six years. We had legislation each year. And it took a long time to go through all the codes. So it was done through all the RCW's. So it was done during the interim when the legislature was not in session. But you might check that just for accuracy. all on table one on the same page the existing term journeyman changing it to journey. You might want to check that when we did the the whole RCW you know all of them we found that there were several terms that were official terms and they had not been changed in the federal codes, chapters, and we did not change those. I'm thinking that Journeyman was one of those, so you might, and I think we could always amend it. Yes, there will be an amendment at full council in those days, because there's other ordinances that are coming along that will be adopted first. Very good. Okay, thank you. Thank you, council member. Any other questions or comments? Before we move to a vote, I'll just observe that I have heard in Common usage people refer to journey level. I don't know if that works in the context that you were looking at And I appreciate taking a look that before we take this up at full council So this is on for action today. If we are prepared to do so I would entertain a motion to approve ordinance 2018 0 0 6 8 with a do pass recommendation It's been moved and seconded by two of my male colleagues. Thank you so much gentlemen All those in favor. Oh, I'm sorry no amendments Any other comments before we move to vote? Will the clerk please call the roll Thank You madam chair councilmember Dembowski councilmember Dunn councilmember Gossett councilmember Colwell councilmember Lambert councilmember McDermott councilmember up the grove councilmember von Reichbauer Madam chair aye madam chair the vote is seven ayes no noes to excused All right the motion passes And we'll be on the regular course to give some time for those amendments to catch up not on consent Not on consent, and it actually is required to have a public hearing notice, so it'll be out okay a few weeks very good All right, let's go back to item number two, and if you're ready mark could you please call the roll? Thank You madam chair councilmember Dembalski Council Member Dunn? Aye. Council Member Gossett? Aye. Council Member Cole-Wells? You're here. Yeah. Somebody started us off on that pattern. Good morning and welcome to the committee of the whole. Council Member Lambert? Council Member McDermott? Here. Council Member Upthegrove? Council Member Von Rechbauer? Here. Madam Chair? Here. Madam chair you have a quorum thank you and if councilmember McDermott would you please move approval of the minutes for January 17 so moved madam chair it's been moved it is before us to approve the minutes of our January 17th meeting any comments or changes seeing none all those in favor please signify by saying aye aye any opposed the minutes are approved all right that brings us to the last two items on our agenda both of which will need to be discussed in executive session. The first is an item on the convention center, the second is on the implementation of ordinance 18-403. The grounds for the executive session under RCW 42.30.110 are to discuss with legal counsel litigation or potential litigation to which the county is or is likely to become a party when public knowledge regarding the discussion is likely to result in an adverse legal or financial consequence to the county. We'll be an executive session on these two items for approximately 90 minutes until about well let's just call it 93 minutes until about 11 o'clock so I did the math all wrong we're gonna until about 11 o'clock let's just leave it at that so I'm gonna ask the clerk to post the doors to that effect we're gonna have to ask the public to leave the chambers at this time and since we're gonna take up the Convention Center first I'm gonna ask anyone not directly related to that Discussion to also, please leave the chamber until we move on to the next agenda item so let's move into executive session you can see I'm gonna have to ask you to step out, please Mr.. Nebbe hi, I'm gonna have to ask any member of the public to please step out. We're going into executive session. Thank you Thank you for coming today Thank you very much for joining us and we'll see you next time. I think I'm going to have to go to the other side of the room and see if I can find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find a way to find public knowledge of the discussion could have an effect an adverse effect on the county's consideration of the matter thank you very well so that's for the Convention Center and the reason I stated previously will be the grounds for executive session on ordinance 18 403 we're gonna do them one and then the other and then we will adjourn from executive session so at this time let's post the doors and go into executive session thank you for helping me clean up the record very good all right

Good morning and welcome to the February 7th, 2018 meeting of the King County Council Committee of the Whole. Today, the committee will be discussing the second of four ordinances to make the King County Code gender neutral, and then we will be going into an extended studies executive session for two items, one regarding the convention center sale, the other regarding the implementation of Ordinance 18403, which set compensation fees and costs for using county right-of-way. And we will probably not come, I'm predicting at this point, we will not come out of executive session, we'll just adjourn from executive session. So those are the topics on the agenda for today. I will skip the role until we have a few more people, and go right to public comment. I think as you all know, the Committee of the Whole offers the public the opportunity to make comments on any item on today's agenda. I've just talked through what those are. Each person will have two minutes to speak. Public comment must address an item on today's agenda. Public comment may not be used for the purpose of assisting a campaign for election of any person or for the promotion or opposition of any ballot proposition. It must not include obscene speech. If a speaker fails to abide by these restrictions, I will rule them out of order, require them to return to their seat. So with that, we have one person signed up to give public comment, and that's Mr. Alex Zimmerman. Good morning and welcome. Good morning. How are you doing, gentlemen? I'm not a social democratic mafia with progressive Gestapo principle. My name is Alex Zimmerman and I'm president of Stand Up America. I want to speak about this agenda because agenda is make me absolutely don't understandable. So you right now change from Virginia and penis to somebody who like an ordinary man. I'm totally confused about this. It's a huge ordinary like a hundred probably. Yeah, it's good changes. My question right now, very simple. When you have fundamental changes like this, why you have fundamental changes for 30% taxes? What is we have only for one calendar year now? I don't think so this happen before. So when we change from man to woman to something, people, you know what it mean? It's okay with me, but how we can change to 30% taxes is I cannot understand this. If you're talking about billion and billion dollars. So we need pay right now for sound transit, hundred billion dollars Panzers have. We need pay right now for teacher, five billion dollars another not real changes in Boeing, nine billion dollars right now in another billion and billion dollars. You never come changes to normal situation. When you change from man to woman into people, I understand this. But when you cannot change hundred billion dollars, what is you stolen from us like a criminal, like a gangster, like a killer, I cannot understand this. Small changes very good too, but we need right now come back to America. What is my president Donald Trump talking about this? And he call you a low life. Mr. Zimmerman, are you going to speak to something on the agenda today? Exactly I speak about this. I'm not hearing a single thing that's related to anything that's on the agenda. You have too many changes, but you never have a fundamental change. What is make us life better. Fundamental changes that make our life better is not on the agenda today. Stand up America. You're out of time. Thank you. All right, I understand that we have one other person who would like to speak and that's Kasich Zenebe. Welcome. I'll ask you also, please speak to something on the agenda today. That's what public comment is for. So if you could please try to speak to something on our agenda. And the items are about gender neutral code, the convention center sale, and compensation for fees and costs using King County right of way. Thank you. Thank you. All right, we're going to move on. We'll also pass over item four and approve the minutes when we have a quorum and move to item five, which is proposed ordinance number 2018-0068. It's our continuing work to establish a gender neutral language in our code. This one has a number of titles, which I assume will be listed out. I have them here, but I'm not going to list them out. And it's the second of four planned ordinances that together will make the entire King County code gender neutral as we last year have the went to the voters to make the King County charter gender neutral. We have King County central staff, Aaron Osens here to present. Welcome. And please go ahead and give your presentation. Good morning, Aaron Osens, council staff. The materials for this item begin on page nine of your packet. Proposed ordinance 2018-0068 is the second in a series of four ordinances that would make changes to the King County code to remove gendered pronouns and historically gendered terms wherever possible. This ordinance includes the titles of the King County code that collectively are the county's development regulations, which do include titles 9, 13, 14, 16, 17, 19A, 20, 21A, 23, 27 and 27A. No substantive legal or policy changes are proposed to be made as part of this ordinance, but other drafting corrections are proposed by the code revisor and have been incorporated into the proposed ordinance. By way of background in July, 2016, motion 14680 was passed by the council directing the clerk of the council to develop options for how to apply gender neutral references throughout the King County code. The same day related ordinance 18316 placed an item on the ballot to amend the charter to make the language of the charter gender neutral. That charter amendment passed by a majority of the voters in November of 2016. This proposed ordinance is consistent with existing Washington state law and county code to be written in gender neutral terms. In the proposed ordinance, gendered pronouns such as he, him, she or her are replaced with the title of the actor in impacted sentences. For example, in sections that refer to the director as he or she, the proposed ordinance changes the gendered pronoun to the director, thereby naming the title of the actor and disregarding the gender. Table one on page 10 of the packet contains a sample of other proposed changes to historically gendered terms. A comprehensive list of the gendered terms addressed in the proposed ordinance is available in attachment two of the staff report, which starts on page 159 of the packet. Executive staff were consulted regarding the proposed changes and their feedback has been addressed and incorporated into the proposed ordinance. In attachment three on page 163 of your packet is the timeline for the future ordinances. There are two additional ordinances that will be presented to the council and that will then complete the review of the remaining titles of the code. The entire project should be complete by May of 2018. That concludes my report. As with the previous ordinance, it's important to recognize the work of Russell Pethel and Bruce Ritzin from the clerk's office on this project. Thank you, Erin. And you as well. The staff has been doing an amazing job on this. It's a lot of detail work, but I think it's really, we're going to have a better code at the end of it. I really appreciated the portion of your report that talked about some of the detail that goes into this and the practicality where we had proposed changing manhole, as in manhole covers, to like utility access or some point. But apparently out on the streets and in documents, there are abbreviations all over the place that say MH, so we came up with maintenance hole so that now people will know what MH means. That's a good example of incorporating the executive staff's comments, so that was their request. So I appreciate the level of effort that's going into making this work well. Any questions or comments? Council Member Colwells. Thank you, Madam Chair. Yes, Erin and all the staff, thank you for all your excellent work on this. Two things I'd like to bring up. On the top of page 10, just so we'll have accurate information here, on the second line, additionally, in 2007, basically the state legislature passed Senate Bill 5063. That was the first year of, I believe, it was six years. We had legislation each year, and it took a long time to go through all the codes. So it was done through all the RCWs. So it was done during the interim when the legislature was not in session. But you might check that just for accuracy. Second of all, on table one, on the same page, the existing term journeymen changing it to journey. You might want to check that. When we did the whole RCW, you know, all of them, we found that there were several terms that were official terms, and they had not been changed in the federal codes, chapters. And we did not change those. I'm thinking that journeymen was one of those. So you might, and I think we could always amend it. Yes, there will be an amendment at full council on this because there's other ordinances that are coming along that will be adopted first. Very good. Okay, thank you. Thank you, Council Member. Any other questions or comments? Before we move to a vote, I'll just observe that I have heard in common usage people refer to journey level. I don't know if that works in the context that you were looking at. And I appreciate you taking a look at that before we take this up at full council. So this is on for action today. If we are prepared to do so, I would entertain a motion to approve ordinance 2018-0068 with a do pass recommendation. Council Member. It's been moved and seconded by two of my male colleagues. Thank you so much, gentlemen. All those in favor, please signify by saying aye. Aye


### analysis of transcription

In [15]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a KingCounty council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [16]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [17]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [18]:
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a KingCounty council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
Good morning and welcome to the February 7th, 2018 meeting of the King County Council Committee of the Whole. Today, the committee will be discussing the second of four ordinances to make the King County Code gender neutral, and then we will be going into an extended studies executive session for two items, one regarding the convention center sale, the other regarding the implementation of Ordinance 18403

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Aye<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Summary**
Attendees: King County Council Committee of the Whole
Location: King County Council Chambers
Date: February 7th, 2018

The King County Council Committee of the Whole met to discuss the second of four ordinances to make the King County Code gender neutral. The committee also went into an extended studies executive session for two items, one regarding the convention center sale and the other regarding the implementation of Ordinance 18403.

**Discussion Points**

* Public comment was given by two individuals, Alex Zimmerman and Kasich Zenebe, regarding the agenda items.
* Proposed ordinance 2018-0068 was presented by Aaron Osens, council staff, to make changes to the King County code to remove gendered pronouns and historically gendered terms.
* The proposed ordinance includes titles 9, 13, 14, 16, 17, 19A, 20, 21A, 23, 27, and 27A, which collectively are the county's development regulations.
* The ordinance includes

In [19]:
response = tokenizer.decode(outputs[0])
display(Markdown(response))


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a KingCounty council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
Good morning and welcome to the February 7th, 2018 meeting of the King County Council Committee of the Whole. Today, the committee will be discussing the second of four ordinances to make the King County Code gender neutral, and then we will be going into an extended studies executive session for two items, one regarding the convention center sale, the other regarding the implementation of Ordinance 18403, which set compensation fees and costs for using county right-of-way. And we will probably not come, I'm predicting at this point, we will not come out of executive session, we'll just adjourn from executive session. So those are the topics on the agenda for today. I will skip the role until we have a few more people, and go right to public comment. I think as you all know, the Committee of the Whole offers the public the opportunity to make comments on any item on today's agenda. I've just talked through what those are. Each person will have two minutes to speak. Public comment must address an item on today's agenda. Public comment may not be used for the purpose of assisting a campaign for election of any person or for the promotion or opposition of any ballot proposition. It must not include obscene speech. If a speaker fails to abide by these restrictions, I will rule them out of order, require them to return to their seat. So with that, we have one person signed up to give public comment, and that's Mr. Alex Zimmerman. Good morning and welcome. Good morning. How are you doing, gentlemen? I'm not a social democratic mafia with progressive Gestapo principle. My name is Alex Zimmerman and I'm president of Stand Up America. I want to speak about this agenda because agenda is make me absolutely don't understandable. So you right now change from Virginia and penis to somebody who like an ordinary man. I'm totally confused about this. It's a huge ordinary like a hundred probably. Yeah, it's good changes. My question right now, very simple. When you have fundamental changes like this, why you have fundamental changes for 30% taxes? What is we have only for one calendar year now? I don't think so this happen before. So when we change from man to woman to something, people, you know what it mean? It's okay with me, but how we can change to 30% taxes is I cannot understand this. If you're talking about billion and billion dollars. So we need pay right now for sound transit, hundred billion dollars Panzers have. We need pay right now for teacher, five billion dollars another not real changes in Boeing, nine billion dollars right now in another billion and billion dollars. You never come changes to normal situation. When you change from man to woman into people, I understand this. But when you cannot change hundred billion dollars, what is you stolen from us like a criminal, like a gangster, like a killer, I cannot understand this. Small changes very good too, but we need right now come back to America. What is my president Donald Trump talking about this? And he call you a low life. Mr. Zimmerman, are you going to speak to something on the agenda today? Exactly I speak about this. I'm not hearing a single thing that's related to anything that's on the agenda. You have too many changes, but you never have a fundamental change. What is make us life better. Fundamental changes that make our life better is not on the agenda today. Stand up America. You're out of time. Thank you. All right, I understand that we have one other person who would like to speak and that's Kasich Zenebe. Welcome. I'll ask you also, please speak to something on the agenda today. That's what public comment is for. So if you could please try to speak to something on our agenda. And the items are about gender neutral code, the convention center sale, and compensation for fees and costs using King County right of way. Thank you. Thank you. All right, we're going to move on. We'll also pass over item four and approve the minutes when we have a quorum and move to item five, which is proposed ordinance number 2018-0068. It's our continuing work to establish a gender neutral language in our code. This one has a number of titles, which I assume will be listed out. I have them here, but I'm not going to list them out. And it's the second of four planned ordinances that together will make the entire King County code gender neutral as we last year have the went to the voters to make the King County charter gender neutral. We have King County central staff, Aaron Osens here to present. Welcome. And please go ahead and give your presentation. Good morning, Aaron Osens, council staff. The materials for this item begin on page nine of your packet. Proposed ordinance 2018-0068 is the second in a series of four ordinances that would make changes to the King County code to remove gendered pronouns and historically gendered terms wherever possible. This ordinance includes the titles of the King County code that collectively are the county's development regulations, which do include titles 9, 13, 14, 16, 17, 19A, 20, 21A, 23, 27 and 27A. No substantive legal or policy changes are proposed to be made as part of this ordinance, but other drafting corrections are proposed by the code revisor and have been incorporated into the proposed ordinance. By way of background in July, 2016, motion 14680 was passed by the council directing the clerk of the council to develop options for how to apply gender neutral references throughout the King County code. The same day related ordinance 18316 placed an item on the ballot to amend the charter to make the language of the charter gender neutral. That charter amendment passed by a majority of the voters in November of 2016. This proposed ordinance is consistent with existing Washington state law and county code to be written in gender neutral terms. In the proposed ordinance, gendered pronouns such as he, him, she or her are replaced with the title of the actor in impacted sentences. For example, in sections that refer to the director as he or she, the proposed ordinance changes the gendered pronoun to the director, thereby naming the title of the actor and disregarding the gender. Table one on page 10 of the packet contains a sample of other proposed changes to historically gendered terms. A comprehensive list of the gendered terms addressed in the proposed ordinance is available in attachment two of the staff report, which starts on page 159 of the packet. Executive staff were consulted regarding the proposed changes and their feedback has been addressed and incorporated into the proposed ordinance. In attachment three on page 163 of your packet is the timeline for the future ordinances. There are two additional ordinances that will be presented to the council and that will then complete the review of the remaining titles of the code. The entire project should be complete by May of 2018. That concludes my report. As with the previous ordinance, it's important to recognize the work of Russell Pethel and Bruce Ritzin from the clerk's office on this project. Thank you, Erin. And you as well. The staff has been doing an amazing job on this. It's a lot of detail work, but I think it's really, we're going to have a better code at the end of it. I really appreciated the portion of your report that talked about some of the detail that goes into this and the practicality where we had proposed changing manhole, as in manhole covers, to like utility access or some point. But apparently out on the streets and in documents, there are abbreviations all over the place that say MH, so we came up with maintenance hole so that now people will know what MH means. That's a good example of incorporating the executive staff's comments, so that was their request. So I appreciate the level of effort that's going into making this work well. Any questions or comments? Council Member Colwells. Thank you, Madam Chair. Yes, Erin and all the staff, thank you for all your excellent work on this. Two things I'd like to bring up. On the top of page 10, just so we'll have accurate information here, on the second line, additionally, in 2007, basically the state legislature passed Senate Bill 5063. That was the first year of, I believe, it was six years. We had legislation each year, and it took a long time to go through all the codes. So it was done through all the RCWs. So it was done during the interim when the legislature was not in session. But you might check that just for accuracy. Second of all, on table one, on the same page, the existing term journeymen changing it to journey. You might want to check that. When we did the whole RCW, you know, all of them, we found that there were several terms that were official terms, and they had not been changed in the federal codes, chapters. And we did not change those. I'm thinking that journeymen was one of those. So you might, and I think we could always amend it. Yes, there will be an amendment at full council on this because there's other ordinances that are coming along that will be adopted first. Very good. Okay, thank you. Thank you, Council Member. Any other questions or comments? Before we move to a vote, I'll just observe that I have heard in common usage people refer to journey level. I don't know if that works in the context that you were looking at. And I appreciate you taking a look at that before we take this up at full council. So this is on for action today. If we are prepared to do so, I would entertain a motion to approve ordinance 2018-0068 with a do pass recommendation. Council Member. It's been moved and seconded by two of my male colleagues. Thank you so much, gentlemen. All those in favor, please signify by saying aye. Aye<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Summary**
Attendees: King County Council Committee of the Whole
Location: King County Council Chambers
Date: February 7th, 2018

The King County Council Committee of the Whole met to discuss the second of four ordinances to make the King County Code gender neutral. The committee also went into an extended studies executive session for two items, one regarding the convention center sale and the other regarding the implementation of Ordinance 18403.

**Discussion Points**

* Public comment was given by two individuals, Alex Zimmerman and Kasich Zenebe, regarding the agenda items.
* Proposed ordinance 2018-0068 was presented by Aaron Osens, council staff, to make changes to the King County code to remove gendered pronouns and historically gendered terms.
* The proposed ordinance includes titles 9, 13, 14, 16, 17, 19A, 20, 21A, 23, 27, and 27A, which collectively are the county's development regulations.
* The ordinance includes drafting corrections proposed by the code revisor and has been incorporated into the proposed ordinance.
* Council Member Colwell brought up a correction to the proposed ordinance regarding the state legislature passing Senate Bill 5063 and the term "journeymen" in table one.

**Takeaways**

* The King County Code will be made gender neutral through a series of four ordinances.
* The proposed ordinance includes drafting corrections proposed by the code revisor and has been incorporated into the proposed ordinance.
* The county's development regulations will be updated to remove gendered pronouns and historically gendered terms.

**Action Items**

* Approve ordinance 2018-0068 with a do pass recommendation.
* Owner: King County Council Committee of the Whole
* Status: Approved
* Next steps: The ordinance will be presented to the full council for adoption.

**Additional Action Items**

* Check the accuracy of the information on page 10 of the packet regarding the state legislature passing Senate Bill 5063.
* Check the term "journeymen" in table one to ensure it is accurate.
* Owner: Council Member Colwell<|eot_id|>